## Notebook 08: Semantic Matching Labeling

This notebook creates a labeled dataset for evaluating the quality of semantic matching between ecoinvent activities and PSILCA sectors. It benchmarks multiple embedding models and a TF-IDF baseline, and provides an interactive labeling interface.

**What it does:**
1. Loads PSILCA sector names and a sample of ecoinvent activity descriptors.
2. Embeds both sides using multiple models (MiniLM, BGE, MPNet) and a TF-IDF baseline.
3. Retrieves the top-K most similar PSILCA sectors for each activity and model.
4. Writes an audit workbook with candidate rankings.
5. Provides an ipywidgets UI for manual labeling of plausible matches.

**Inputs:**
- PSILCA sector list JSON (from SENSE pipeline output).
- Ecoinvent activity list JSON (from SENSE pipeline output).

**Outputs:**
- `semantic_matching_audit.xlsx`: unlabeled audit workbook with top-K candidates per model.
- `semantic_matching_audit_labeled.xlsx`: labeled workbook after manual review.

In [ ]:
# %% [markdown]
# # Semantic matching audit — Notebook A (labeling)
#
# Creates an Excel workbook with top-K sector candidates for each activity and model.
# Then provides an optional ipywidgets UI to label "plausible" ranks.
#
# Outputs (strictly necessary):
# - semantic_matching_audit.xlsx
# - semantic_matching_audit_labeled.xlsx (after labeling)

# %%
from __future__ import annotations

import os
import json
import hashlib
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Baseline (lexical) model
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except Exception as e:
    raise ImportError(
        "scikit-learn is required for the TF-IDF baseline. "
        "Install with: pip install scikit-learn"
    ) from e

# %% [markdown]
# ## 0) User inputs

# %%
# =========================
# USER INPUTS (EDIT THESE)
# =========================
PSILCA_SECTORS_JSON = r"PSILCA_SECTORS_JSON"
ECOINVENT_ACTIVITIES_JSON = r"ECOINVENT_ACTIVITIES_JSON"

OUT_DIR = r"OUT_DIR"
os.makedirs(OUT_DIR, exist_ok=True)

# Sample and splitting
RANDOM_SEED = 42
LOW_RES_MAX = 26
N_HIGH_TOTAL = 150
N_LOW_TOTAL = 100
DEV_FRAC_HIGH = 0.60
DEV_FRAC_LOW = 0.60
MAX_PER_DESCRIPTOR_PER_STRATUM = 1  # cap duplicate descriptors in each stratum

# Candidate list sizes written to the audit workbook
TOP_K_FOR_LABELING = 10
INCLUDE_TOP50_REFERENCE = True      # set False to reduce file size
TOP_K_FOR_REFERENCE = 50

# ---- Lexical baseline (TF-IDF cosine; character n-grams) ----
TFIDF_BASELINE = "baseline/tfidf_char_wb_3_5"
TFIDF_ANALYZER = "char_wb"     # robust for partial overlaps + tokenization differences
TFIDF_NGRAM_RANGE = (3, 5)     # common strong baseline

# Models to benchmark
MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-base-en-v1.5",
    "sentence-transformers/all-mpnet-base-v2",
    TFIDF_BASELINE,  # <-- baseline added
]

# Embedding settings
BATCH_SIZE = 256
DEVICE = None  # e.g., "cpu" or "cuda"

# Labeling UI behavior
SYNC_ACROSS_MODELS = True  # set False if you want to avoid syncing identical sectors across models

print("OUT_DIR:", OUT_DIR)
print("MODELS:", MODELS)

# %% [markdown]
# ## 1) Model-specific text preprocessing (BGE prompts)

# %%
BGE_MODEL = "BAAI/bge-base-en-v1.5"
BGE_MODE = "intended"  # "intended" or "tool_legacy"

BGE_QUERY_PROMPT = "Represent this sentence for searching relevant passages: "
DOC_PROMPT_TOOL = "Represent this text for retrieval: "

def prep_plain(text: str) -> str:
    return (text or "").strip()

def prep_bge_query_intended(text: str) -> str:
    return BGE_QUERY_PROMPT + (text or "").strip()

def prep_tool_legacy(text: str) -> str:
    return DOC_PROMPT_TOOL + (text or "").strip()

def get_preprocessors(model_name: str):
    """
    Returns (prep_activity, prep_sector) for the given model.
    """
    if model_name == BGE_MODEL:
        if BGE_MODE == "intended":
            return prep_bge_query_intended, prep_plain
        if BGE_MODE == "tool_legacy":
            return prep_tool_legacy, prep_tool_legacy
        raise ValueError("BGE_MODE must be 'intended' or 'tool_legacy'")
    return prep_plain, prep_plain

print("BGE_MODE:", BGE_MODE)

# %% [markdown]
# ## 2) Load inputs and flatten ecoinvent activities

# %%
def read_json(path: str) -> Any:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

psilca_sectors: Dict[str, List[str]] = read_json(PSILCA_SECTORS_JSON)
ecoinvent_raw = read_json(ECOINVENT_ACTIVITIES_JSON)

# normalize PSILCA country keys to upper ISO2
psilca_sectors = {k.strip().upper(): v for k, v in psilca_sectors.items()}

sector_count = {c: len(v) for c, v in psilca_sectors.items()}
print(f"Loaded PSILCA countries: {len(psilca_sectors)}")
print(f"Loaded top-level ecoinvent records: {len(ecoinvent_raw)}")
print("PSILCA sector count (min/median/max):",
      min(sector_count.values()),
      int(np.median(list(sector_count.values()))),
      max(sector_count.values()))

def normalize_mapping(x: Any) -> Optional[List[str]]:
    """
    Return list of ISO2 codes (upper) or None if invalid.
    """
    if x is None:
        return None
    if isinstance(x, str):
        s = x.strip().upper()
        return [s] if len(s) == 2 else None
    if isinstance(x, list):
        cleaned = [v.strip().upper() for v in x if isinstance(v, str) and len(v.strip()) == 2]
        return cleaned if cleaned else None
    return None

rows: List[Dict[str, Any]] = []
for rec in ecoinvent_raw:
    mapped = normalize_mapping(rec.get("PSILCA_mapping")) or normalize_mapping(rec.get("ISO2"))
    for a in (rec.get("Activities") or []):
        rows.append({
            "ecoinvent_iso2": (rec.get("ISO2") or "").strip(),
            "psilca_countries": mapped,
            "name": a.get("name") or "",
            "reference_product": a.get("reference product") or "",
            "unit": a.get("unit") or "",
            "code": a.get("code") or "",
            "isic": a.get("ISIC rev.4 ecoinvent") or "",
            "price": a.get("price"),
        })

df = pd.DataFrame(rows)
print("Flattened activities:", df.shape)

# Keep only activities mapped to exactly one PSILCA country that exists in PSILCA
df["n_mapped"] = df["psilca_countries"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df1 = df[df["n_mapped"] == 1].copy()
df1["psilca_country"] = df1["psilca_countries"].apply(lambda x: x[0])
df1 = df1[df1["psilca_country"].isin(psilca_sectors.keys())].copy()

df1["n_sectors"] = df1["psilca_country"].map(sector_count).astype(int)

def make_descriptor(row: pd.Series) -> str:
    """
    Exact descriptor template used for matching (lowercase).
    """
    name = (row["name"] or "").strip()
    ref = (row["reference_product"] or "").strip()
    isic = (row["isic"] or "").strip()
    return f"{name}; {ref}; isic rev.4: {isic}".lower().strip()

df1["descriptor"] = df1.apply(make_descriptor, axis=1)
print("Eligible (single-country) activities:", df1.shape)

# %% [markdown]
# ## 3) Stratified sample + dev/test split (+ duplicate cap)

# %%
def stable_group_seed(base_seed: int, group_name: str) -> int:
    h = hashlib.sha1(f"{base_seed}||{group_name}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)

def cap_duplicates(df_in: pd.DataFrame, key_col: str, cap: int, base_seed: int) -> pd.DataFrame:
    """
    For each unique key, keep at most `cap` rows (deterministic).
    """
    def _sample_group(g: pd.DataFrame) -> pd.DataFrame:
        if len(g) <= cap:
            return g
        rs = stable_group_seed(base_seed, str(g.name))
        return g.sample(n=cap, random_state=rs)

    return (
        df_in.groupby(key_col, group_keys=False)
            .apply(_sample_group)
            .reset_index(drop=True)
    )

df_low = df1[df1["n_sectors"] <= LOW_RES_MAX].copy()
df_high = df1[df1["n_sectors"] > LOW_RES_MAX].copy()

df_high_cap = cap_duplicates(df_high, "descriptor", MAX_PER_DESCRIPTOR_PER_STRATUM, RANDOM_SEED)
df_low_cap  = cap_duplicates(df_low,  "descriptor", MAX_PER_DESCRIPTOR_PER_STRATUM, RANDOM_SEED + 1)

if len(df_high_cap) < N_HIGH_TOTAL:
    raise ValueError(f"Not enough high-res after cap; need {N_HIGH_TOTAL}, have {len(df_high_cap)}")
if len(df_low_cap) < N_LOW_TOTAL:
    raise ValueError(f"Not enough low-res after cap; need {N_LOW_TOTAL}, have {len(df_low_cap)}")

high_sample = df_high_cap.sample(n=N_HIGH_TOTAL, random_state=RANDOM_SEED).reset_index(drop=True)
low_sample  = df_low_cap.sample(n=N_LOW_TOTAL, random_state=RANDOM_SEED + 1).reset_index(drop=True)

# shuffle within stratum before split assignment for determinism
high_sample = high_sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
low_sample  = low_sample.sample(frac=1, random_state=RANDOM_SEED + 1).reset_index(drop=True)

n_high_dev = int(round(N_HIGH_TOTAL * DEV_FRAC_HIGH))
n_low_dev  = int(round(N_LOW_TOTAL * DEV_FRAC_LOW))

high_sample["split"] = ["dev"] * n_high_dev + ["test"] * (N_HIGH_TOTAL - n_high_dev)
low_sample["split"]  = ["dev"] * n_low_dev  + ["test"] * (N_LOW_TOTAL - n_low_dev)

df_sample = pd.concat([high_sample, low_sample], ignore_index=True)
df_sample = df_sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

def make_activity_id(code: str, country: str) -> str:
    # code is typically unique; country included for stability/traceability
    s = f"{code}|{country}"
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:10]

df_sample["activity_id"] = df_sample.apply(lambda r: make_activity_id(r["code"], r["psilca_country"]), axis=1)
df_sample["stratum"] = np.where(df_sample["n_sectors"] <= LOW_RES_MAX, "low_res", "high_res")

print(df_sample[["split", "stratum"]].value_counts())

# %% [markdown]
# ## 4) Score + rank sectors (embeddings + TF-IDF baseline; in-memory, no caching)

# %%
def embed_texts(model: SentenceTransformer, texts: List[str], preprocess) -> np.ndarray:
    texts_clean = [preprocess(t) for t in texts]
    emb = model.encode(
        texts_clean,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return emb.astype(np.float32)

def topk_indices(scores: np.ndarray, k: int) -> np.ndarray:
    """
    Return indices (n,k) sorted by descending score.
    """
    n = scores.shape[1]
    k = min(k, n)
    part = np.argpartition(-scores, kth=k-1, axis=1)[:, :k]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(-part_scores, axis=1)
    return np.take_along_axis(part, order, axis=1)

def safe_model_key(model_name: str) -> str:
    return model_name.replace("/", "__").replace(":", "_")

def is_tfidf_model(model_name: str) -> bool:
    return model_name == TFIDF_BASELINE

def tfidf_scores_for_country(activity_texts: List[str], sector_texts: List[str]) -> np.ndarray:
    """
    TF-IDF cosine similarity (dot product under L2 normalization),
    using character n-grams. Returns dense scores array shape (n_act, n_sec).
    """
    vec = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        ngram_range=TFIDF_NGRAM_RANGE,
        lowercase=True,
    )
    corpus = sector_texts + activity_texts
    X = vec.fit_transform(corpus)  # sparse
    X_sec = X[:len(sector_texts)]
    X_act = X[len(sector_texts):]
    S = (X_act @ X_sec.T)          # sparse (n_act, n_sec)
    return S.toarray().astype(np.float32)

all_results: Dict[str, Dict[str, pd.DataFrame]] = {}

countries_in_sample = sorted(df_sample["psilca_country"].unique().tolist())
print("Countries in sample:", len(countries_in_sample))

for model_name in MODELS:
    print("\nModel:", model_name)

    rows_top10 = []
    rows_top50 = []

    if not is_tfidf_model(model_name):
        model = SentenceTransformer(model_name, device=DEVICE)
        prep_act, prep_sec = get_preprocessors(model_name)

        # Activity embeddings (sample only)
        act_emb = embed_texts(model, df_sample["descriptor"].tolist(), preprocess=prep_act)

        # Sector embeddings per country (compute once per country, reuse for its activities)
        sector_emb_by_country: Dict[str, Tuple[List[str], np.ndarray]] = {}
        for c in countries_in_sample:
            sector_names = list(psilca_sectors[c])
            sec_emb = embed_texts(model, sector_names, preprocess=prep_sec)
            sector_emb_by_country[c] = (sector_names, sec_emb)

    # score by country group
    for country, grp in tqdm(df_sample.groupby("psilca_country"), desc=f"Scoring ({safe_model_key(model_name)})"):
        grp = grp.sort_index()  # stable order
        sector_names = list(psilca_sectors[country])

        idxs = grp.index.to_numpy()

        if is_tfidf_model(model_name):
            # TF-IDF baseline: compute scores directly for this group's descriptors
            act_texts = grp["descriptor"].astype(str).tolist()
            scores = tfidf_scores_for_country(act_texts, sector_names)  # (n_act, n_sec)
        else:
            # Embedding models: cosine similarity (normalized embeddings)
            _, sector_emb = sector_emb_by_country[country]
            scores = act_emb[idxs] @ sector_emb.T  # (n_act, n_sec)

        k10 = min(TOP_K_FOR_LABELING, scores.shape[1])
        top10_idx = topk_indices(scores, k10)

        k50 = min(TOP_K_FOR_REFERENCE, scores.shape[1])
        top50_idx = topk_indices(scores, k50) if INCLUDE_TOP50_REFERENCE else None

        for r_i, act_index in enumerate(idxs):
            aid = df_sample.at[act_index, "activity_id"]
            row = {
                "model": model_name,
                "activity_id": aid,
                "split": df_sample.at[act_index, "split"],
                "psilca_country": country,
                "n_sectors": int(df_sample.at[act_index, "n_sectors"]),
                "name": df_sample.at[act_index, "name"],
                "reference_product": df_sample.at[act_index, "reference_product"],
                "isic": df_sample.at[act_index, "isic"],
                "descriptor": df_sample.at[act_index, "descriptor"],
                "plausible_ranks": "",
                "fail": "",
                "notes": "",
            }

            for j in range(k10):
                sidx = int(top10_idx[r_i, j])
                row[f"sector_{j+1}"] = sector_names[sidx]
                row[f"score_{j+1}"] = float(scores[r_i, sidx])
            for j in range(k10, TOP_K_FOR_LABELING):
                row[f"sector_{j+1}"] = ""
                row[f"score_{j+1}"] = np.nan

            rows_top10.append(row)

        if INCLUDE_TOP50_REFERENCE and top50_idx is not None:
            for r_i, act_index in enumerate(idxs):
                aid = df_sample.at[act_index, "activity_id"]
                for j in range(k50):
                    sidx = int(top50_idx[r_i, j])
                    rows_top50.append({
                        "model": model_name,
                        "activity_id": aid,
                        "split": df_sample.at[act_index, "split"],
                        "psilca_country": country,
                        "n_sectors": int(df_sample.at[act_index, "n_sectors"]),
                        "rank": j+1,
                        "sector": sector_names[sidx],
                        "score": float(scores[r_i, sidx]),
                    })

    df_top10 = (pd.DataFrame(rows_top10)
                .sort_values(["split", "psilca_country", "activity_id"])
                .reset_index(drop=True))
    if INCLUDE_TOP50_REFERENCE:
        df_top50 = (pd.DataFrame(rows_top50)
                    .sort_values(["split", "psilca_country", "activity_id", "rank"])
                    .reset_index(drop=True))
    else:
        df_top50 = pd.DataFrame()

    all_results[model_name] = {"top10": df_top10, "top50": df_top50}

print("Done. Models processed:", len(all_results))

# %% [markdown]
# ## 5) Write audit workbook (semantic_matching_audit.xlsx)

# %%
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.datavalidation import DataValidation

audit_xlsx = os.path.join(OUT_DIR, "semantic_matching_audit.xlsx")

wb = Workbook()
wb.remove(wb.active)

# README
ws = wb.create_sheet("README")
readme_lines = [
    "Semantic matching audit workbook",
    "",
    "Purpose:",
    "- Benchmark candidate sector matching models on a stratified activity sample.",
    "- Label whether any of the top-K suggested sectors is plausible.",
    "",
    "Models included:",
    "- Sentence-transformer bi-encoders (cosine similarity)",
    f"- Lexical baseline: {TFIDF_BASELINE} (TF-IDF cosine; analyzer={TFIDF_ANALYZER}, ngram_range={TFIDF_NGRAM_RANGE})",
    "",
    "Labeling instructions (per model *_top10 sheet):",
    f"1) Inspect sector_1..sector_{TOP_K_FOR_LABELING} (and descriptor).",
    "2) If 1+ sectors are plausible, enter their ranks in 'plausible_ranks' e.g.: 1,3,7",
    "3) If none are plausible, leave plausible_ranks empty (FAIL is optional).",
    "4) Optional notes in 'notes'.",
    "",
    "After labeling, save a copy as semantic_matching_audit_labeled.xlsx",
]
for i, line in enumerate(readme_lines, start=1):
    ws.cell(row=i, column=1, value=line)

# hidden list for FAIL validation (optional)
ws_list = wb.create_sheet("lists")
ws_list["A1"] = "FAIL"
ws_list.sheet_state = "hidden"

# sample sheet for traceability (kept small; useful for SI)
ws_s = wb.create_sheet("sample")
sample_cols = ["activity_id","split","psilca_country","n_sectors","code","name",
               "reference_product","unit","isic","price","descriptor"]
for r in dataframe_to_rows(df_sample[sample_cols], index=False, header=True):
    ws_s.append(r)
ws_s.freeze_panes = "A2"

for model_name in MODELS:
    key = safe_model_key(model_name)
    df_top10 = all_results[model_name]["top10"]
    df_top50 = all_results[model_name]["top50"]

    ws10 = wb.create_sheet(f"{key}_top10")
    for r in dataframe_to_rows(df_top10, index=False, header=True):
        ws10.append(r)
    ws10.freeze_panes = "A2"

    # add FAIL dropdown validation (optional)
    headers = [c.value for c in ws10[1]]
    fail_col = headers.index("fail") + 1
    dv_fail = DataValidation(type="list", formula1="=lists!$A$1:$A$1", allow_blank=True)
    ws10.add_data_validation(dv_fail)
    dv_fail.add(f"{ws10.cell(row=2, column=fail_col).coordinate}:{ws10.cell(row=ws10.max_row, column=fail_col).coordinate}")

    if INCLUDE_TOP50_REFERENCE and len(df_top50):
        ws50 = wb.create_sheet(f"{key}_top50")
        for r in dataframe_to_rows(df_top50, index=False, header=True):
            ws50.append(r)
        ws50.freeze_panes = "A2"

wb.save(audit_xlsx)
print("Wrote:", audit_xlsx)

# %% [markdown]
# ## 6) Optional: labeling UI (ipywidgets)
#
# If you prefer labeling directly in Excel, you can stop here.
# Otherwise, run this section and click through the activities.

# %%
import math
import ipywidgets as widgets
from IPython.display import display
from openpyxl import load_workbook

BASE_XLSX   = audit_xlsx
OUTPUT_XLSX = os.path.join(OUT_DIR, "semantic_matching_audit_labeled.xlsx")
STATE_JSON  = os.path.join(OUT_DIR, "_label_state_widgets.json")

TOP_K = TOP_K_FOR_LABELING

def load_top10_tables(xlsx_path: str) -> Dict[str, pd.DataFrame]:
    xls = pd.ExcelFile(xlsx_path)
    out = {}
    for m in MODELS:
        sheet = f"{safe_model_key(m)}_top10"
        out[m] = pd.read_excel(xlsx_path, sheet_name=sheet).set_index("activity_id")
    return out

top10_tables = load_top10_tables(BASE_XLSX)
ref_model = MODELS[0]
activity_ids = (
    top10_tables[ref_model].reset_index()
    .sort_values(["split","psilca_country","activity_id"])["activity_id"]
    .astype(str).tolist()
)

def load_state() -> Dict[str, Any]:
    st = {"idx": 0, "labels": {m: {} for m in MODELS}}
    if os.path.exists(STATE_JSON):
        try:
            with open(STATE_JSON, "r", encoding="utf-8") as f:
                loaded = json.load(f)
            st["idx"] = int(loaded.get("idx", 0))
            lab = loaded.get("labels", {})
            if isinstance(lab, dict):
                for m in MODELS:
                    if isinstance(lab.get(m), dict):
                        st["labels"][m] = lab[m]
            print(f"✅ Resumed at idx={st['idx']}")
        except Exception as e:
            print("⚠️ Could not load state, starting fresh:", e)
    st["idx"] = max(0, min(st["idx"], len(activity_ids)-1))
    return st

def save_state(st: Dict[str, Any]) -> None:
    with open(STATE_JSON, "w", encoding="utf-8") as f:
        json.dump(st, f, indent=2)

state = load_state()

def norm_sector(s: str) -> str:
    return " ".join((s or "").split()).casefold()

IS_LOADING = False
SYNCING = False

def current_aid() -> str:
    return activity_ids[state["idx"]]

# --- UI scaffolding
title = widgets.HTML(value="")
meta = widgets.HTML(value="")
descriptor = widgets.Textarea(value="", layout=widgets.Layout(width="100%", height="110px"))
status = widgets.HTML(value="")

model_cols: Dict[str, Dict[str, Any]] = {}

def update_fail_display(m: str) -> None:
    any_avail = any((not cb.disabled) for cb in model_cols[m]["checks"])
    any_sel = any(cb.value for cb in model_cols[m]["checks"] if not cb.disabled)
    fail = (any_avail and not any_sel) or (not any_avail)
    model_cols[m]["fail_lbl"].value = "<span style='color:red; font-weight:700;'>FAIL</span>" if fail else ""

def sync_sector_across_models(sector_key: str, new_value: bool) -> None:
    global SYNCING
    if (not SYNC_ACROSS_MODELS) or (not sector_key):
        return
    SYNCING = True
    try:
        for m2 in MODELS:
            for j in range(TOP_K):
                if model_cols[m2]["sector_keys"][j] == sector_key:
                    model_cols[m2]["checks"][j].value = new_value
    finally:
        SYNCING = False
    for m2 in MODELS:
        update_fail_display(m2)

def make_observer(model_name: str, idx: int):
    def _obs(change):
        if change.get("name") != "value":
            return
        if IS_LOADING or SYNCING:
            return
        key = model_cols[model_name]["sector_keys"][idx]
        if key:
            sync_sector_across_models(key, bool(change["new"]))
    return _obs

# dynamic column width
col_width = f"{max(20, int(100/len(MODELS)))}%"

for m in MODELS:
    checks, labels, rows = [], [], []
    for i in range(TOP_K):
        cb = widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width="24px"))
        lab = widgets.HTML(value="", layout=widgets.Layout(width="100%"))
        checks.append(cb)
        labels.append(lab)
        rows.append(widgets.HBox([cb, lab], layout=widgets.Layout(align_items="flex-start")))

    notes = widgets.Text(value="", placeholder="optional notes", layout=widgets.Layout(width="100%"))
    fail_lbl = widgets.HTML(value="")

    box = widgets.VBox(
        [widgets.HTML(f"<b>{m}</b>")] + rows + [fail_lbl, widgets.HTML("<b>Notes (optional):</b>"), notes],
        layout=widgets.Layout(border="1px solid #ddd", padding="8px", width=col_width)
    )

    model_cols[m] = {
        "checks": checks,
        "labels": labels,
        "notes": notes,
        "fail_lbl": fail_lbl,
        "box": box,
        "sector_keys": [""] * TOP_K,
    }

    for i, cb in enumerate(checks):
        cb.observe(make_observer(m, i), names="value")

def store_current_to_state() -> None:
    aid = current_aid()
    for m in MODELS:
        ranks = [i+1 for i, cb in enumerate(model_cols[m]["checks"]) if cb.value]
        notes_val = model_cols[m]["notes"].value or ""
        state["labels"].setdefault(m, {})
        state["labels"][m][aid] = {"ranks": ranks, "notes": notes_val}
    save_state(state)

def load_activity(idx: int) -> None:
    global IS_LOADING
    IS_LOADING = True
    try:
        state["idx"] = max(0, min(idx, len(activity_ids)-1))
        aid = current_aid()

        r = top10_tables[ref_model].loc[aid]
        title.value = f"<h3 style='margin:0'>Activity {state['idx']+1}/{len(activity_ids)} — {aid}</h3>"
        meta.value = (
            f"<div style='font-family:monospace'>"
            f"split={r.get('split','')} | country={r.get('psilca_country','')} | n_sectors={r.get('n_sectors','')}<br>"
            f"<b>name</b>={r.get('name','')}<br>"
            f"<b>ref</b>={r.get('reference_product','')}<br>"
            f"<b>isic</b>={r.get('isic','')}"
            f"</div>"
        )
        descriptor.value = str(r.get("descriptor","") or "")

        for m in MODELS:
            row = top10_tables[m].loc[aid]
            saved = state["labels"].get(m, {}).get(aid, {"ranks": [], "notes": ""})
            saved_ranks = set(saved.get("ranks", []))
            model_cols[m]["notes"].value = saved.get("notes", "")

            for i in range(TOP_K):
                sec = str(row.get(f"sector_{i+1}", "") or "").strip()
                score = row.get(f"score_{i+1}", None)

                cb = model_cols[m]["checks"][i]
                lab = model_cols[m]["labels"][i]
                model_cols[m]["sector_keys"][i] = norm_sector(sec) if sec else ""

                if sec:
                    cb.disabled = False
                    score_txt = ""
                    if score is not None and not (isinstance(score, float) and math.isnan(score)):
                        score_txt = f"<span style='color:#666'>(score={float(score):.3f})</span>"
                    lab.value = f"<div style='white-space:normal; line-height:1.2;'><b>{i+1}.</b> {sec} {score_txt}</div>"
                    cb.value = (i+1) in saved_ranks
                else:
                    cb.disabled = True
                    cb.value = False
                    lab.value = "<div style='color:#999'>(no sector)</div>"

        for m in MODELS:
            update_fail_display(m)

        save_state(state)
        status.value = "<span style='color:#666'>Autosaved state.</span>"
    finally:
        IS_LOADING = False

def ranks_to_str(ranks: List[int]) -> str:
    return ",".join(str(x) for x in sorted(set(int(r) for r in ranks))) if ranks else ""

def save_to_excel() -> str:
    store_current_to_state()
    wb = load_workbook(BASE_XLSX)

    for m in MODELS:
        sheet = f"{safe_model_key(m)}_top10"
        ws = wb[sheet]
        headers = [cell.value for cell in ws[1]]
        col = {name: headers.index(name) + 1 for name in ["activity_id", "plausible_ranks", "fail", "notes"]}

        # map activity_id -> row index
        row_map = {}
        for r in range(2, ws.max_row + 1):
            aid = ws.cell(r, col["activity_id"]).value
            if aid:
                row_map[str(aid)] = r

        for aid, rec in state["labels"].get(m, {}).items():
            if aid not in row_map:
                continue
            rr = row_map[aid]
            ranks = rec.get("ranks", [])
            ws.cell(rr, col["plausible_ranks"]).value = ranks_to_str(ranks)
            ws.cell(rr, col["fail"]).value = "FAIL" if len(ranks) == 0 else ""
            ws.cell(rr, col["notes"]).value = rec.get("notes", "")

    wb.save(OUTPUT_XLSX)
    return OUTPUT_XLSX

btn_prev = widgets.Button(description="◀ Prev")
btn_next = widgets.Button(description="Next ▶", button_style="primary")
btn_save = widgets.Button(description="Save to Excel", button_style="success")

def on_prev(_):
    store_current_to_state()
    load_activity(state["idx"] - 1)

def on_next(_):
    store_current_to_state()
    load_activity(state["idx"] + 1)

def on_save(_):
    path = save_to_excel()
    status.value = f"<span style='color:green; font-weight:700'>Saved:</span> {path}"

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_save.on_click(on_save)

display(widgets.VBox([
    title,
    meta,
    widgets.HTML("<b>Descriptor</b>"),
    descriptor,
    widgets.HTML("<hr>"),
    widgets.HBox([model_cols[m]["box"] for m in MODELS]),
    widgets.HBox([btn_prev, btn_next, btn_save]),
    status
]))

load_activity(state["idx"])
